## Entwicklung eines Knowledge Graphs zur Analyse und Visualisierung von Metadaten in der Sicherheitsforschung für automatisierte und autonome Fahrsysteme
### Bachelorarbeit von Heike Schlitt im Studiengang Wirtschaftsinformatik 
### am Lehrstuhl Datenbanken und Informationssysteme der Fakultät Mathematik und Informatik an der FernUniversität in Hagen

Ziel dieser Arbeit ist es die von Westhofen et al. entwickelte Ontologie zu Kritikalitätsphänomenen (A.U.T.O.) im urbanen Straßenverkehr mit Hilfe eines LLMs mit natürlichsprachlichen Unfallbeschreibungen in Verbindung zu setzen und die Unfallbeschreibungen auf diese Kritikalitätsphänomene zu analysieren. Die identifizierten Phänomene sollen in einem Knowledge Graph festgehalten, auf ihre Kardinalitäten analysiert und visualisiert werden. Zudem soll versucht werden Kritikalitätsphänomene zu identifizieren, die nicht in der Ontologie enthalten sind, um die Ontologie ggfs. erweitern zu können.


In [1]:
# Importblock
import yaml
from google import genai
from PyPDF2 import PdfReader    


In [2]:
# Konfiguration (Quellen, Prompt, AI- Modell)
PDF_FILE_PATH = "G:\\Meine Ablage\\FernUni\\Bachelorarbeit\\Unfallbeschreibungen\\NHTSA\\TECHNICAL DOCUMENT\\RPT970071316.PDF"
OWL_AUTO_FILE_PATH = "G:\\Meine Ablage\\FernUni\\Bachelorarbeit\\Ontologien\\automotive_urban_traffic_ontology.owl"
OWL_CP_FILE_PATH = "G:\\Meine Ablage\\FernUni\\Bachelorarbeit\\Ontologien\\criticality_phenomena.owl"
OWL_CPF_FILE_PATH = "G:\\Meine Ablage\\FernUni\\Bachelorarbeit\\Ontologien\\criticality_phenomena_formalization.owl"

USER_QUESTION = "Die 3 .txt-Dateien bilden zusammen eine Ontologie zur Beschreibung und Formalisierung von Kritikalitätsphänomenen im urbanen Straßenverkehr. Identifiziere anhand der A.U.T.O. die Unfallursachen aus dem PDF-Dokument, sowie die Zusammenhänge und Beziehungen der Vorgänge, die zu dem Unfall geführt haben. Ignoriere dabei Verletzungen und Ursachen die innerhalb der Fahrzeuge lokalisiert sind. Ordne die Unfallursachen, wenn möglich, den durch die A.U.T.O. gelieferten Metadaten zu. Sollte keine Zudordnung möglich sein, erstelle eine Kategorie 'neue Kritikalitäsphänomene'. Die Daten, die du den Metadaten zurodnen kannst, gebe zusätzlich als Cypher-Code an, damit ein Knowledge Graph mit der selben 3-teiligen Struktur und Syntax der A.U.T.O. erstellt werden kann."

MODEL_NAME = "gemini-1.5-flash" # gemini-1.5-pro, gemini-2.5-pro-preview-03-25

In [3]:
# YAML-Datei laden
with open("config.yaml", "r") as file:
    config = yaml.safe_load(file)

API_KEY = config["GoogleAI"]["API-Key"]

# Client mit API-Schlüssel erstellen
client = genai.Client(api_key=API_KEY)

# PDF einlesen / gesamten Text extrahieren
def extract_text_from_pdf(pdf_file_path):
    """
    Extrahiert den gesamten Text aus einer mehrseitigen PDF-Datei.
    
    :param pdf_file_path: Pfad zur PDF-Datei
    :return: Gesamter Text der PDF-Datei als String
    """
    reader = PdfReader(pdf_file_path)
    all_text = ""
    for page in reader.pages:
        all_text += page.extract_text() + "\n"  # Text jeder Seite hinzufügen
    return all_text

pdf_text = extract_text_from_pdf(PDF_FILE_PATH)

# OWL-Dateien als String einlesen
with open(OWL_AUTO_FILE_PATH, "r") as file:
    owl_auto_content = file.read()
with open(OWL_CP_FILE_PATH, "r") as file:
    owl_cp_content = file.read()
with open(OWL_CPF_FILE_PATH, "r") as file:
    owl_cpf_content = file.read()

In [7]:
# Übergabe der Daten an die API zur Verarbeitung und Antwortgenerierung
print(f"\nSending request to Gemini API (Model: {MODEL_NAME})...")

response = client.models.generate_content(
    model=MODEL_NAME,
    contents=f"{USER_QUESTION}\n\nZusätzlicher Dateiinhalt:\n{pdf_text, owl_auto_content, owl_cp_content, owl_cpf_content}",)


Sending request to Gemini API (Model: gemini-1.5-flash)...


ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'The model is overloaded. Please try again later.', 'status': 'UNAVAILABLE'}}

In [ ]:
# Ausgabe der LLM-Antwort
print("\n--- Gemini Response ---")
if response.text:
    print(response.text)
else:
    print("Warning: Received an empty response or the response might have been blocked.")
    print("Full response object for debugging:")
    print(response)
    if response.prompt_feedback:
        print(f"Prompt Feedback: {response.prompt_feedback}")


--- Gemini Response ---
Der Unfallbericht beschreibt einen Unfall zwischen einem Dodge Caravan und einem Ford F-150 Pickup Truck bei nassen Straßenverhältnissen.  Die Unfallursachen, die *außerhalb* der Fahrzeuge liegen und  mit der A.U.T.O. in Verbindung gebracht werden können, sind:

**1. Wet Road Conditions:**

* **A.U.T.O. Klasse:** `CP_202` (Wet Road)
* **Cypher:** `(c:CP_202 {id: "CA03-007_WetRoad"})`

Der Bericht erwähnt explizit, dass die Fahrbahn nass war. Dies beeinflusste die Reifenhaftung und trug somit zur Unfalldynamik bei.

**2. Erratic Driving of Dodge Caravan:**

* **A.U.T.O. Klasse:** `CP_142` (Non-Ego-TP Unexpected Driving) -  da das Verhalten als "erratic" und "fishtailing" beschrieben wird, deutet dies auf unerwartetes Fahrverhalten hin. Eine genauere Klassifizierung ist aufgrund der beschränkten Informationen schwierig.
* **Cypher:** `(c:CP_142 {id: "CA03-007_ErraticDriving", description: "Dodge Caravan fishtailing"})`

Der Bericht deutet auf ein möglicherweise f